# Llama3.1 Function Calling

see: https://llama.meta.com/docs/model-cards-and-prompt-formats/llama3_1/

In [1]:
model_id = "llama3.1:8b"

In [2]:
# Using Ollama
# !pip install ollama
import ollama

In [18]:
client = ollama.Client()
# Initialize conversation with a user query
messages = [{'role': 'user', 'content': 'What is the flight time from New York (NYC) to Los Angeles (LAX)?'}]

tools = [{
                'type': 'function',
                'function': {
                  'name': 'get_flight_times',
                  'description': 'Get the flight times between two cities',
                  'parameters': {
                    'type': 'object',
                    'properties': {
                      'departure': {
                        'type': 'string',
                        'description': 'The departure city (airport code)',
                      },
                      'arrival': {
                        'type': 'string',
                        'description': 'The arrival city (airport code)',
                      },
                    },
                    'required': ['departure', 'arrival'],
                  },
                },
              }]


# First API call: Send the query and function description to the model
response = client.chat(
                model=model_id,
                messages=messages,
                tools=tools,
                )
response['message'].get('tool_calls',[])

[{'function': {'name': 'get_flight_times',
   'arguments': {'arrival': 'LAX', 'departure': 'NYC'}}}]

In [19]:
for key in response.keys():
    print(key,":",response[key])

model : llama3.1:8b
created_at : 2024-09-14T15:56:53.219161Z
message : {'role': 'assistant', 'content': '', 'tool_calls': [{'function': {'name': 'get_flight_times', 'arguments': {'arrival': 'LAX', 'departure': 'NYC'}}}]}
done_reason : stop
done : True
total_duration : 1175772708
load_duration : 27752666
prompt_eval_count : 189
prompt_eval_duration : 651307000
eval_count : 26
eval_duration : 495835000


In [23]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_delivery_date",
            "description": "Get the delivery date for a customer's order. Call this whenever you need to know the delivery date, for example when a customer asks 'Where is my package'",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The customer's order ID.",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,
            },
        }
    }
]

messages = [
    {"role": "system", "content": "You are a helpful customer support assistant. Use the supplied tools to assist the user."},
    {"role": "user", "content": "Hi, can you tell me the delivery date for my order with id 4711?"}
]

response = client.chat(
                model=model_id,
                messages=messages,
                tools=tools,
                )
response['message'].get('tool_calls',[])



[{'function': {'name': 'get_delivery_date',
   'arguments': {'order_id': '4711'}}}]

In [24]:
response

{'model': 'llama3.1:8b',
 'created_at': '2024-09-14T15:59:49.87024Z',
 'message': {'role': 'assistant',
  'content': '',
  'tool_calls': [{'function': {'name': 'get_delivery_date',
     'arguments': {'order_id': '4711'}}}]},
 'done_reason': 'stop',
 'done': True,
 'total_duration': 736347084,
 'load_duration': 28207084,
 'prompt_eval_count': 214,
 'prompt_eval_duration': 340120000,
 'eval_count': 20,
 'eval_duration': 365730000}

In [35]:
########### With OpenAI API #############
from openai import OpenAI
client = OpenAI(
    base_url = 'http://localhost:11434/v1',
    api_key='ollama', # required, but unused
)

response = client.chat.completions.create(
    model=model_id,
    messages=messages,
    tools=tools,
)
print(response.choices[0].message.tool_calls[0])

ChatCompletionMessageToolCall(id='call_ibnzg91u', function=Function(arguments='{"order_id":"4711"}', name='get_delivery_date'), type='function')


In [93]:
########### Backup ###########

In [94]:
## Function Calling with TinyAgent

# see: https://github.com/SqueezeAILab/TinyAgent
import ollama

model_id = "andthattoo/tinyagent-1.1b:latest"

client = ollama.Client()

messages = [{'role': 'user', 'content': 'My name is Dominik and my job title is partner. Compose a new email to Bob and sent it tomorrow at 12pm.'}]
response = client.chat(model=model_id, messages=messages)

response

{'model': 'andthattoo/tinyagent-1.1b:latest',
 'created_at': '2024-09-07T12:13:27.895198Z',
 'message': {'role': 'assistant', 'content': ''},
 'done_reason': 'stop',
 'done': True,
 'total_duration': 100351083,
 'load_duration': 8693375,
 'prompt_eval_count': 57,
 'prompt_eval_duration': 90353000,
 'eval_count': 1,
 'eval_duration': 11000}